# 02. Document Processing  
— Docling 기반 Structure-aware Chunking

## 실습 목적

`01_naive_rag_limits.ipynb`에서는 **Docling으로 PDF를 읽은 뒤 `export_to_text()`로 평탄화하고,  
`RecursiveCharacterTextSplitter`로 분할**하는 Naive RAG 구조를 사용했다.  

이번 실습에서는 Docling이 인식한 **문서 구조와 provenance metadata를 버리지 않고**  
`HybridChunker`를 이용해 구조 인식 청크를 만든다.

```text
01 Naive RAG
PDF
 → Docling
 → Plain Text
 → Recursive Chunking
 → Dense Retrieval

02 Document Processing
PDF
 → Docling Document
 → Structure-aware / Token-aware Chunking
 → Metadata
 → Dense Retrieval
```

## 학습 목표

실습 후 다음 내용을 설명할 수 있어야 한다.

1. PDF를 Markdown/Text 중간 파일로 변환하지 않고 Docling 문서 구조를 직접 사용하는 이유
2. `HybridChunker`가 계층 구조와 token 크기를 함께 고려하는 방식
3. heading, caption, page provenance를 chunk metadata에 보존하는 방법
4. metadata가 검색 결과 추적과 이후 citation에 필요한 이유
5. Naive RAG의 문자 길이 기반 chunking과 구조 인식 chunking의 차이

## 실습 데이터

`data/AI@Data_Report_2026_전망_251223(최종).pdf`


## 1. 환경 설정

필요 패키지 예시:

```bash
uv add docling langchain-core langchain-chroma langchain-openai tiktoken python-dotenv
uv add "docling-core[chunking-openai]"
```

> `HybridChunker`에는 문자열이 아니라 tokenizer 객체를 전달한다.  
> 이 실습에서는 이후 임베딩 모델과 동일한 `text-embedding-3-small` 계열 tokenizer를 사용한다.


In [1]:
import sys
# 상위 폴더의 공통 설정(common_config.py)을 import 할 수 있도록 경로 추가
sys.path.append("..")
# 임베딩 모델 / LLM 연결 함수 (공통 설정)
from common_config import embedding_model, llm_connect

from pathlib import Path
# 실습 대상 PDF (노트북 기준 data/ 폴더)
PDF_PATH = Path("data/AI@Data_Report_2026_전망_251223(최종).pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")

print(f"PDF: {PDF_PATH}")


PDF: data/AI@Data_Report_2026_전망_251223(최종).pdf


# 2. Docling으로 PDF 구조 분석

Docling의 `DocumentConverter`가 생성한 `DoclingDocument`를 그대로 사용한다.  
이렇게 해야 제목, 표, 문단, provenance 등의 구조 정보를 chunking 단계까지 전달할 수 있다.

PDF heading hierarchy 복원을 활성화하여 문서의 절/소절 구조를 최대한 보존한다.

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import HeadingHierarchyOptions, PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# 이 보고서는 텍스트 기반 PDF이므로 OCR은 끈다.
# 스캔 PDF를 사용할 경우 do_ocr=True로 변경할 수 있다.
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False

# 표의 행/열/셀 구조까지 인식해 표 단위로 보존
pipeline_options.do_table_structure = True

# PDF의 제목 계층을 추론하도록 설정
# 제목 계층(절/소절)을 추론해 heading 구조를 만든다
pipeline_options.heading_hierarchy_options = HeadingHierarchyOptions(enabled=True)
# PDF를 파싱한 페이지별 중간 결과(parsed page)를 메모리에 보관하도록 켜는 옵션
pipeline_options.generate_parsed_pages = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

# PDF → DoclingDocument 변환 (텍스트 평탄화 없이 구조를 가진 객체로 받음)
conversion_result = converter.convert(PDF_PATH)
# 이후 chunking 단계에 그대로 넘길 구조화 문서 객체
docling_doc = conversion_result.document

print("Docling 변환 완료")
print(f"문서명: {PDF_PATH.name}")


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Docling 변환 완료
문서명: AI@Data_Report_2026_전망_251223(최종).pdf


## 2.1 Docling이 인식한 문서 구조 확인

`export_to_markdown()`은 구조 확인용으로만 사용한다.

In [3]:
# 구조 확인용 Markdown 미리보기 (chunking에는 사용하지 않음)
md_preview = docling_doc.export_to_markdown()

print(md_preview[:3000])

####### AI@Data Report

###### 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망

2025-제3호

####### 목차

- ❶ 배경 및 목적 /p.06
- ❷ 자료 분석 및 방법 /p.09
- ❸ 분석 결과 /p.12
- ❹ 2026년 전망: AI 생태계 방향성 /p.17
- ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20

<!-- image -->

##### AI@Data Report

2025 제 3 호

(2025. 12월)

####### 토픽 분석을 통한

####### AI 주요 트렌드 및 2026 전망

AI@Data 보고서 시리즈는...

AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.

- ❶ 본 보고서는 과학기술정보통신부와 한국지능정보사회진흥원(NIA)의 '초거대 AI 데이터 발굴 및 기획' 사업의 산출물이므로, 본 보고서의 내용을 발표할 때는 반드시 과학기술정보통신부 사업의 연구결과임을 밝혀야합니다.
- ❷ 보고서 내용의 무단전재를 금하며, 가공‧인용할 때는 반드시 출처를 「한국지능정보사회진흥원(NIA)」이라고 밝혀주시기 바랍니다.
- ❸ 본 보고서의 내용은 한국지능정보사회진흥원(NIA)의 공식 견해와 다를 수 있습니다.

작성 한국지능정보사회진흥원 김시연 수석(siyeon@nia.or.kr, 053-230-4249)

추지혜 책임(chuu@nia.or.kr, 053-230-4247)

전민석 주임(msj1224@nia.or.kr, 053-230-4205)

####### 기획 한국지능정보사회진흥원 인공지능데이터본부

신신애 본부장(sashin@nia.or.kr, 053-230-4200)

심호찬 팀장(shc@nia.or.kr, 053-230-4201)

####### 표·그림 목차

| [표1]   | 분석 과정                       

# 3. Structure-aware Chunking

## HybridChunker

`HybridChunker`는 다음 두 관점을 함께 사용한다.

1. **문서 구조**
   - 제목/절의 계층
   - 같은 heading 아래의 문단
   - 표와 caption

2. **token 크기**
   - 너무 큰 구조 단위는 token 한도를 기준으로 추가 분할
   - 같은 heading의 작은 인접 chunk는 가능한 경우 병합

즉, 단순한 `chunk_size=800` 문자 분할과 달리 **의미 있는 문서 구조를 먼저 존중하고,
필요할 때 token 기준으로 조정**한다.


In [ ]:
import tiktoken

from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer

# 임베딩 모델과 동일한 tokenizer를 써서 chunk 길이를 실제 임베딩 기준으로 측정
EMBED_MODEL_NAME = "text-embedding-3-small"

# chunk 1개의 최대 token 수 (검색 단위 크기를 조절하는 값)
MAX_TOKENS = 700

encoding = tiktoken.encoding_for_model(EMBED_MODEL_NAME)

# Docling chunker가 token 수를 계산할 때 쓰는 tokenizer
tokenizer = OpenAITokenizer(
    tokenizer=encoding,
    max_tokens=MAX_TOKENS,
)

# 문서 구조(heading/표/문단) 기준 + token 한도 기준을 함께 사용하는 chunker
chunker = HybridChunker(
    tokenizer=tokenizer,
    # 같은 heading 아래의 작은 인접 chunk를 token 한도 내에서 병합
    merge_peers=True,
    # 표가 여러 chunk로 나뉘어도 각 chunk에 표 헤더를 반복
    repeat_table_header=True,
)

# DoclingDocument 전체를 구조 단위 chunk 목록으로 변환
docling_chunks = list(chunker.chunk(dl_doc=docling_doc))

print(f"생성된 Docling chunk 수: {len(docling_chunks)}")


생성된 Docling chunk 수: 48


## 3.1 Chunk의 구조 정보 확인

Docling chunk에는 본문뿐 아니라 heading, caption, 원본 문서 item의 provenance(출처)가 포함될 수 있다.

특히 page 정보는 이후 다음 용도에 중요하다.

- 검색 결과 디버깅
- 출처 표시
- citation 생성
- 평가용 relevant page 비교


In [5]:
def get_page_numbers(chunk):
    """Docling chunk의 provenance에서 페이지 번호를 추출한다."""
    # 같은 chunk 안의 여러 item이 같은 페이지를 가리킬 수 있으므로 set으로 중복 제거
    pages = {
        prov.page_no
        for item in chunk.meta.doc_items
        for prov in getattr(item, "prov", [])
        if getattr(prov, "page_no", None) is not None
    }
    return sorted(pages)


for i, chunk in enumerate(docling_chunks[:5], start=1):
    pages = get_page_numbers(chunk)
    headings = chunk.meta.headings or []
    captions = chunk.meta.captions or []

    print("=" * 100)
    print(f"[Chunk {i}]")
    print(f"pages    : {pages}")
    print(f"headings : {headings}")
    print(f"captions : {captions}")
    print("-" * 100)
    print(chunk.text[:800])
    print()


[Chunk 1]
pages    : [1]
headings : ['토픽 분석을 통한 AI 주요 트렌드 및 2026 전망']
captions : []
----------------------------------------------------------------------------------------------------
2025-제3호

[Chunk 2]
pages    : [1]
headings : ['토픽 분석을 통한 AI 주요 트렌드 및 2026 전망', '목차']
captions : []
----------------------------------------------------------------------------------------------------
- ❶ 배경 및 목적 /p.06
- ❷ 자료 분석 및 방법 /p.09
- ❸ 분석 결과 /p.12
- ❹ 2026년 전망: AI 생태계 방향성 /p.17
- ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20

[Chunk 3]
pages    : [2]
headings : ['AI@Data Report']
captions : []
----------------------------------------------------------------------------------------------------
2025 제 3 호
(2025. 12월)

[Chunk 4]
pages    : [2, 3]
headings : ['AI@Data Report', 'AI 주요 트렌드 및 2026 전망']
captions : []
----------------------------------------------------------------------------------------------------
AI@Data 보고서 시리즈는...
AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥

/tmp/ipykernel_139995/2076271392.py:16: DeprecationWarning: deprecated
  captions = chunk.meta.captions or []


# 4. LangChain Document로 변환하면서 Metadata 정규화

Docling의 풍부한 metadata를 그대로 넣으면 nested object가 포함될 수 있다.
Vector DB에서는 검색·필터링하기 쉬운 **단순한 metadata 필드**로 정규화하는 것이 좋다.

이번 실습에서는 다음 정보를 저장한다.

- `source`: 원본 PDF 경로
- `chunk_id`: chunk 식별자
- `page`: 대표 페이지
- `pages`: chunk가 걸쳐 있는 전체 페이지
- `heading`: 문서 heading breadcrumb
- `caption`: 표/그림 caption
- `num_tokens`: contextualized chunk의 token 수
- `parser`: Docling 사용 여부
- `chunker`: chunking 방식

Embedding에는 `chunk.text`만 넣지 않고 `chunker.contextualize(chunk)`를 사용한다.
이를 통해 heading/caption 같은 문맥이 검색 표현에 반영될 수 있다.


In [6]:
from langchain_core.documents import Document

chunks = []

for chunk_id, chunk in enumerate(docling_chunks, start=1):
    pages = get_page_numbers(chunk)
    headings = chunk.meta.headings or []
    captions = chunk.meta.captions or []

    # 검색용 텍스트: heading/caption context가 반영된 문자열
    # 검색(임베딩)용 텍스트: heading/caption 문맥이 붙은 문자열 (chunk.text만 쓰는 것보다 검색에 유리)
    contextualized_text = chunker.contextualize(chunk)

    metadata = {
        "source": str(PDF_PATH),
        "chunk_id": chunk_id,
        # 대표 페이지(첫 페이지), 페이지 정보가 없으면 -1
        "page": pages[0] if pages else -1,
        # chunk가 걸친 전체 페이지를 문자열로 저장 (Vector DB metadata는 list보다 단순 타입이 안전)
        "pages": ",".join(map(str, pages)),
        "heading": " > ".join(map(str, headings)),
        "caption": " | ".join(map(str, captions)),
        # 실제 임베딩 입력(contextualized text) 기준 token 수
        "num_tokens": tokenizer.count_tokens(contextualized_text),
        "parser": "docling",
        "chunker": "HybridChunker",
    }

    chunks.append(
        Document(
            page_content=contextualized_text,
            metadata=metadata,
        )
    )

print(f"LangChain Document 수: {len(chunks)}")


LangChain Document 수: 48


/tmp/ipykernel_139995/2969402239.py:8: DeprecationWarning: deprecated
  captions = chunk.meta.captions or []


## 4.1 Metadata 확인


In [7]:
for doc in chunks[:5]:
    print("=" * 100)
    print(doc.metadata)
    print("-" * 100)
    print(doc.page_content[:600])
    print()


{'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'chunk_id': 1, 'page': 1, 'pages': '1', 'heading': '토픽 분석을 통한 AI 주요 트렌드 및 2026 전망', 'caption': '', 'num_tokens': 37, 'parser': 'docling', 'chunker': 'HybridChunker'}
----------------------------------------------------------------------------------------------------
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
2025-제3호

{'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'chunk_id': 2, 'page': 1, 'pages': '1', 'heading': '토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 > 목차', 'caption': '', 'num_tokens': 142, 'parser': 'docling', 'chunker': 'HybridChunker'}
----------------------------------------------------------------------------------------------------
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
목차
- ❶ 배경 및 목적 /p.06
- ❷ 자료 분석 및 방법 /p.09
- ❸ 분석 결과 /p.12
- ❹ 2026년 전망: AI 생태계 방향성 /p.17
- ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20

{'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'chunk_id': 3, 'page': 2, 'pages': '2', 'heading': 'AI@Data Report', 'caption': '', 'num_tokens': 

## 4.2 Chunk 크기 통계 확인

구조 인식 chunking이라고 해서 모든 chunk 길이가 같지는 않다.
오히려 제목/표/문단의 의미 단위를 우선 보존하므로 token 수가 다양하게 나타나는 것이 자연스럽다.


In [8]:
from statistics import mean, median

# chunk별 token 수 분포 확인 (지나치게 작거나 큰 chunk 점검용)
token_counts = [doc.metadata["num_tokens"] for doc in chunks]

print(f"chunk 수      : {len(token_counts)}")
print(f"최소 token    : {min(token_counts)}")
print(f"최대 token    : {max(token_counts)}")
print(f"평균 token    : {mean(token_counts):.1f}")
print(f"중앙값 token  : {median(token_counts):.1f}")


chunk 수      : 48
최소 token    : 20
최대 token    : 696
평균 token    : 370.6
중앙값 token  : 370.5


# 5. ChromaDB 구성

이번 과정에서는 검색 조건을 비교하기 쉽도록 cosine distance를 사용한다.

또한 실습 중 chunking 설정을 변경할 수 있으므로 `RESET_DB=True`일 때 기존 DB를 삭제하고 재생성한다.


In [9]:
import shutil

from langchain_chroma import Chroma

# 실습용 Chroma 저장 경로와 컬렉션 이름
DB_PATH = Path("chroma_db/ai_trend_report_docling")
COLLECTION_NAME = "ai_trend_report_2026_docling"

# True면 기존 DB를 지우고 새로 만든다 (chunking 설정을 바꾸면 반드시 재생성)
RESET_DB = True

if RESET_DB and DB_PATH.exists():
    shutil.rmtree(DB_PATH)
    print(f"기존 DB 삭제: {DB_PATH}")

# 공통 설정에서 만든 임베딩 모델 사용
embeddings = embedding_model()

# Document 목록을 임베딩해 Chroma에 저장 (metadata도 함께 저장)
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(DB_PATH),
    # 거리 측정 방식을 cosine으로 지정 (값이 작을수록 유사)
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"저장된 chunk 수: {vector_store._collection.count()}")


기존 DB 삭제: chroma_db/ai_trend_report_docling
저장된 chunk 수: 48


# 6. 구조 인식 검색 결과 확인

`similarity_search_with_score()`의 score는 여기서는 **cosine distance**이다.

- 값이 작을수록 질문과 검색 문서가 가깝다.
- 검색 결과와 함께 page / heading metadata를 확인한다.


In [10]:
def inspect_retrieval(query: str, k: int = 5):
    # 질문과 가장 가까운 chunk k개를 (Document, distance) 쌍으로 조회
    results = vector_store.similarity_search_with_score(query, k=k)

    print("=" * 100)
    print(f"[질문] {query}")
    print(f"[검색 개수] k={k}")
    print("cosine distance: 낮을수록 더 유사함")
    print("=" * 100)

    for rank, (doc, score) in enumerate(results, start=1):
        # 검색된 chunk가 어느 페이지/heading에서 왔는지 확인 (출처 추적)
        meta = doc.metadata

        print(
            f"\n[{rank}] distance={score:.4f} "
            f"| chunk_id={meta.get('chunk_id')} "
            f"| pages={meta.get('pages')}"
        )
        print(f"heading={meta.get('heading')}")
        if meta.get("caption"):
            print(f"caption={meta.get('caption')}")
        print("-" * 100)
        print(doc.page_content[:700].replace("\n", " "))


## 6.1 제목/절 구조가 중요한 질문


In [11]:
query = "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"
inspect_retrieval(query, k=5)


[질문] 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
[검색 개수] k=5
cosine distance: 낮을수록 더 유사함

[1] distance=0.3376 | chunk_id=10 | pages=5
heading=AI@Data Report > ■ 2026년 AI 생태계 방향성 전망
----------------------------------------------------------------------------------------------------
AI@Data Report ■ 2026년 AI 생태계 방향성 전망 - 산업·기술·정책이 서로 다른 변화 축을 보이지만, 2026년에는 세 변화가 동시에 심화되며 AI 생태계의 구조 재편이 가속화될 가능성이 있음 - Ÿ 산업은 도입 확대, 기술은 기능 고도화, 정책은 책임·안전 중심의 제도화가 각각 진전될 것으로 예상되며, 이러한 흐름은 AI 활용의 범위, 속도, 규율 체계에 모두 영향을 미칠 것으로 전망됨

[2] distance=0.4124 | chunk_id=2 | pages=1
heading=토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 > 목차
----------------------------------------------------------------------------------------------------
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 목차 - ❶ 배경 및 목적 /p.06 - ❷ 자료 분석 및 방법 /p.09 - ❸ 분석 결과 /p.12 - ❹ 2026년 전망: AI 생태계 방향성 /p.17 - ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20

[3] distance=0.4193 | chunk_id=40 | pages=18
heading=4.3 2026년 AI 기술 전망
-------------------------------------------------------------------------------------------

## 6.2 표 구조가 중요한 질문

보고서에는 산업·기술·정책 분야별 데이터 수집 키워드가 표로 정리되어 있다.
`repeat_table_header=True`는 표가 여러 chunk로 나뉠 때 각 chunk가 표의 column 의미를 잃지 않도록 돕는다.


In [12]:
query = "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"
inspect_retrieval(query, k=5)


[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수] k=5
cosine distance: 낮을수록 더 유사함

[1] distance=0.4163 | chunk_id=21 | pages=10
heading=분석 절차 CHAPTER2 > 3 2.1 자료 수집 절차 및 분석 과정 > 결과분석
----------------------------------------------------------------------------------------------------
분석 절차 CHAPTER2 3 2.1 자료 수집 절차 및 분석 과정 결과분석 < 표 2 > 분야별 데이터 수집 키워드  AI산업, 사용 키워드 = AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환. AI기술, 사용 키워드 = AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발. AI정책, 사용 키워드 = AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 [ 그림 1 ] 분야별 데이터셋 구성 예시

[2] distance=0.4842 | chunk_id=19 | pages=9
heading=분석 절차 CHAPTER2 > 3 2.1 자료 수집 절차 및 분석 과정 > 결과분석
----------------------------------------------------------------------------------------------------
분석 절차 CHAPTER2 3 2.1 자료 수집 절차 및 분석 과정 결과분석 토픽·키워드 패턴을 분석해 2025년 AI 주요 이슈와 향후 변화 방향 도출 - 본 분석은 2025년 1월부터 11월까지 매주 산업·기술·정책 세 분야별로 국내 매체 1건, 해외 매체 1건씩 총 6건을 선정하여, 정기적인 기사 및 보고서 스크랩을 통해 분석에 필요한 데이터셋을 누적하여 확보함 - Ÿ 연구기관·글로벌 기술기업·전문지·경제지·정부기관·국제기구 등 1

## 6.3 Metadata를 활용한 검색 결과 추적

단순히 답변만 보는 것이 아니라 **어떤 페이지와 어떤 section에서 근거가 검색되었는지** 확인한다.  
이 metadata는 이후 `07_source_answer_ai_trend.ipynb`의 근거 기반 답변과 citation에 다시 활용할 수 있다.


In [13]:
query = "추론형 데이터는 어떤 유형으로 구분되나요?"

# score 없이 Document만 반환 — metadata(page, heading) 추적에 사용
results = vector_store.similarity_search(query, k=5)

for i, doc in enumerate(results, start=1):
    print(
        f"[{i}] page={doc.metadata.get('page')} "
        f"pages={doc.metadata.get('pages')} "
        f"heading={doc.metadata.get('heading')}"
    )
    print(doc.page_content[:350].replace("\n", " "))
    print()


[1] page=21 pages=21 heading=5.2 AI 기술 토픽으로 본 노력 및 대응 > < 추론형 데이터 종류(예시) >
5.2 AI 기술 토픽으로 본 노력 및 대응 < 추론형 데이터 종류(예시) > ① 단계적 '과정' 추론, 설명 = Ÿ AI가 최종 결론에 도달하는 '과정' 자체를 학습하는 데 중점을 둔 데이터 - 절차적 추론 데이터(CoT), 상호작용적 추론 데이터 등 - (예시) 민원처리 절차의 체크리스트 및 결정 기준. ② '근거' 기반 설명, 설명 = Ÿ AI가 답을 말할 때 어떤 근거를 인용해야 하는지 학습시키기 위한 데이터 - 자연어 설명(NLE) 데이터, 추출적 근거 데이터 등 - (예시) 정책 Q&A 공공데이터 개방률 ? → 포털 통계 페이지 2-3 근거를 인용. ③ '맥락·관계' 이해, 설명 = Ÿ 객체 간의 '관계'나 

[2] page=9 pages=9 heading=분석 절차 CHAPTER2 > 3 2.1 자료 수집 절차 및 분석 과정
분석 절차 CHAPTER2 3 2.1 자료 수집 절차 및 분석 과정 - 각 분야별 담론 구조 도출을 위해 본 연구는 데이터 기반 접근법을 적용하였으며, 그 분석 과정은 다음의 단계별 프로세스에 따라 수행되었음

[3] page=8 pages=8 heading=1.3 분석 한계 및 유의사항
1.3 분석 한계 및 유의사항 - 본 분석은 2025년 1월부터 11월까지 국내·해외 주요 언론 및 공공·산업 데이터를 중심으로 수행되었으며, 기사·보고서·정책 자료의 주기적 스크리닝과 키워드 정제에 기반하고 있음 - Ÿ 해외 비언론 데이터, 최신 산업·기술 사례, 기업 내부 자료 등은 자료 접근성과 범위 한계로 인해 깊이 있게 반영되지 못하였음 - Ÿ 특정 기간(11개월)의 담론 흐름에 기반하므로, 빠르게 변화하는 AI 환경의 시간적 민감성을 전부 포착하기에는 어려움이 존재함 - 산업·기술·정책 각 분야에서 LDA 토픽모델링을 적용할 때, 데이터 규모와 키워드 밀도, - 담론 구조 등을 고려하여 토픽 수(k

# 7. 01번 실습과 비교

| 구분 | 01 Naive RAG | 02 Document Processing |
|---|---|---|
| PDF 처리 | Docling | Docling |
| 변환 방식 | `export_to_text()` | `DoclingDocument` 직접 사용 |
| Chunking | Recursive character | Hybrid / structure-aware |
| 기준 | 문자 길이 | 문서 구조 + token |
| Heading | 대부분 소실 | metadata 및 contextualized text 활용 |
| Table context | chunk 경계에서 약화 가능 | 표 구조 및 header 반복 활용 |
| Page provenance | 제한적 | metadata로 추적 |
| Vector 검색 | Dense | Dense |
| Query 변환 | 없음 | 없음 |
| Reranking | 없음 | 없음 |

## 핵심

이번 단계에서는 아직 Query Rewrite나 Reranking을 적용하지 않는다.

**02번의 목적은 "검색 전에 문서를 더 잘 준비하는 것"이다.**

다음 `03_query_rewriting_ai_trend.ipynb`에서는 동일한 Vector DB를 대상으로
사용자 질문 자체를 검색에 더 적합한 형태로 바꾸는 방법을 학습한다.


# 정리

이번 실습에서 적용한 핵심은 다음과 같다.

1. PDF를 Markdown 중간 파일로 저장하지 않고 Docling으로 직접 구조 분석했다.
2. `DoclingDocument`를 `HybridChunker`에 전달해 structure-aware chunking을 수행했다.
3. embedding 모델 계열 tokenizer를 사용해 token-aware splitting을 적용했다.
4. `contextualize()`를 이용해 heading/caption 문맥을 검색용 텍스트에 반영했다.
5. page, heading, caption, chunk_id 등을 metadata로 정규화했다.
6. 구조 인식 chunk를 ChromaDB에 저장하고 검색 결과의 출처를 추적했다.

이 단계의 목적은 **문서 전처리와 chunk 품질을 개선하여 이후 Advanced RAG 단계의 기반을 만드는 것**이다.
